# Desafío por equipos · Machine Learning aplicado 

**Caso:** NovaRetail · **Siete equipos, siete desafíos distintos.** Cada equipo resuelve el suyo.

Todos los equipos entregan lo mismo: una **tabla de métricas**, la **validación de fuga** y una **recomendación defendible**. Lo que cambia es la complicación de cada caso.

Los datos son sintéticos y se generan desde el código; ningún resultado representa una situación real de ARCA. Trabajen solo con estos datos.

## Paso 0 · Instalación

Corré esta celda primero. Si falta alguna librería, la instala en el mismo Python que ejecuta este notebook; si ya está todo, no hace nada.

In [ ]:
# Instalación autocontenida: instala, en el mismo Python que ejecuta este notebook, lo que falte.
import importlib, importlib.util, subprocess, sys

REQUISITOS = {"numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn", "matplotlib": "matplotlib", "jinja2": "jinja2"}
INSTALAR_XGBOOST = False          # opcional: ponelo en True para sumar XGBoost a la comparación de modelos
if INSTALAR_XGBOOST: REQUISITOS["xgboost"] = "xgboost"

print(f"Kernel en uso: {sys.executable} (Python {sys.version.split()[0]})")
faltan = [pip for modulo, pip in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    print("Instalando:", ", ".join(faltan), "(puede tardar un minuto)")
    base = [sys.executable, "-m", "pip", "install", "-q"]
    try:
        try:
            subprocess.check_call(base + faltan)
        except subprocess.CalledProcessError:       # el Python del sistema suele rechazar instalaciones
            print("Reintentando con --break-system-packages ...")
            subprocess.check_call(base + ["--break-system-packages"] + faltan)
    except Exception:
        raise SystemExit("No se pudo instalar en este kernel. Elegí el kernel .venv (arriba a la derecha) y volvé a correr esta celda.")
    importlib.invalidate_caches()
    print("Listo.")
else:
    print("Todo instalado: podés seguir.")


## Parte 1 · Preparación

Esta celda deja todo listo: genera los clientes, hace **la misma partición de la clase** (mismas filas de entrenamiento y de prueba), entrena la **regresión logística** y define las herramientas que usamos en la clase en vivo (`metricas` y `costos_por_umbral`). Solo correla.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

VIOLETA, VERDE, NARANJA, GRIS, ROJO = "#7C5CF6", "#10D9A0", "#F59E0B", "#8888A8", "#E5484D"
plt.rcParams.update({"figure.figsize": (8, 4), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "font.size": 11, "axes.titleweight": "bold"})
pd.options.display.float_format = "{:,.3f}".format

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.calibration import calibration_curve

COLUMNAS = ["id_cliente", "fecha_alta", "canal", "segmento", "region", "edad", "antiguedad_meses", "compras_12m",
            "ticket_promedio", "dias_desde_ultima_compra", "contactos_soporte_90d", "usa_app", "descuento_promedio",
            "satisfaccion", "motivo_baja", "abandono_90d"]


def generar_novaretail(n=8000, semilla=2025, duplicados=60):
    rng = np.random.default_rng(semilla)
    fecha_corte = pd.Timestamp("2025-12-31")
    canal = rng.choice(["web", "app", "tienda"], size=n, p=[0.40, 0.25, 0.35])
    # composición: web concentra más clientes del segmento Esencial (esto confunde la lectura por canal)
    p_seg = {"web": [0.62, 0.28, 0.10], "app": [0.30, 0.45, 0.25], "tienda": [0.28, 0.42, 0.30]}
    segmento = np.array([rng.choice(["Esencial", "Plus", "Premium"], p=p_seg[c]) for c in canal])
    region = rng.choice(["Centro", "Norte", "Sur", "Litoral", "Cuyo"], size=n, p=[0.30, 0.20, 0.18, 0.17, 0.15])
    edad = np.clip(rng.normal(41, 12, n), 18, 80).round().astype(int)
    alta = pd.Timestamp("2021-01-01") + pd.to_timedelta(rng.integers(0, 1730, n), unit="D")
    antig = ((fecha_corte - alta).days / 30.4).round().astype(int)
    base_compras = np.select([segmento == "Esencial", segmento == "Plus"], [5, 9], 14)
    compras = rng.poisson(base_compras)
    ticket = np.round(np.select([segmento == "Esencial", segmento == "Plus"], [35, 62], 110) * rng.lognormal(0, 0.25, n), 2)
    dias = np.clip(rng.gamma(2.2, 24, n), 1, 365).round().astype(int)
    soporte = rng.poisson(np.where(segmento == "Esencial", 1.2, 0.6))
    usa_app = np.where(canal == "app", 1, (rng.random(n) < 0.35).astype(int))
    descuento = np.clip(rng.normal(np.where(segmento == "Esencial", 0.14, 0.08), 0.05, n), 0, 0.35).round(3)
    sat_real = np.clip(np.round(rng.normal(3.7 - 0.12 * soporte + 0.15 * (segmento == "Premium"), 0.9, n)), 1, 5)

    z_dias = (dias - 60) / 40
    logit = (-2.45
             + 0.85 * (segmento == "Esencial") - 0.45 * (segmento == "Premium")
             + 0.80 * z_dias + 0.30 * soporte - 0.42 * (sat_real - 3.7)
             - 0.50 * usa_app - 0.012 * (antig - 24) - 0.04 * (compras - 8)
             + 0.55 * ((canal == "tienda") & (segmento == "Esencial")))
    p = 1 / (1 + np.exp(-logit))
    abandono = (rng.random(n) < p).astype(int)

    # faltantes: la satisfacción falta más entre quienes abandonan (faltante informativo)
    sat = sat_real.copy()
    sat[rng.random(n) < np.where(abandono == 1, 0.30, 0.09)] = np.nan
    ticket = ticket.astype(float)
    ticket[rng.random(n) < 0.03] = np.nan
    region_s = pd.Series(region, dtype=object)
    region_s[rng.random(n) < 0.02] = np.nan
    # valores imposibles
    edad = edad.astype(float)
    malos = rng.random(n) < 0.004
    edad[malos] = rng.choice([0, 150, -1], size=malos.sum())
    # fuga: el motivo de baja solo existe para quienes ya abandonaron
    motivo = np.where(abandono == 1, rng.choice(["precio", "servicio", "competencia", "mudanza"], size=n), None)

    df = pd.DataFrame({
        "id_cliente": [f"NR{i:06d}" for i in range(1, n + 1)],
        "fecha_alta": alta.strftime("%Y-%m-%d"),
        "canal": canal, "segmento": segmento, "region": region_s,
        "edad": edad, "antiguedad_meses": antig, "compras_12m": compras,
        "ticket_promedio": ticket, "dias_desde_ultima_compra": dias,
        "contactos_soporte_90d": soporte, "usa_app": usa_app,
        "descuento_promedio": descuento, "satisfaccion": sat,
        "motivo_baja": motivo, "abandono_90d": abandono,
    })
    # duplicados: filas repetidas por un error de carga
    if duplicados:
        df = pd.concat([df, df.sample(duplicados, random_state=semilla)], ignore_index=True)
    return df.sample(frac=1, random_state=semilla).reset_index(drop=True)


# --- datos: una fila por cliente, ya limpios (como en la clase en vivo) ---
df = generar_novaretail().drop_duplicates("id_cliente").reset_index(drop=True)
df.loc[(df["edad"] < 18) | (df["edad"] > 100), "edad"] = np.nan
df["satisfaccion_faltante"] = df["satisfaccion"].isna().astype(int)

NUM = ["edad", "antiguedad_meses", "compras_12m", "ticket_promedio", "dias_desde_ultima_compra",
       "contactos_soporte_90d", "usa_app", "descuento_promedio", "satisfaccion", "satisfaccion_faltante"]
CAT = ["canal", "segmento", "region"]
X, y = df[NUM + CAT], df["abandono_90d"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
y_prueba = y_test.to_numpy()

# --- pipeline y regresión logística: el modelo base de la clase ---
def armar_pipeline(modelo, num=NUM, cat=CAT):
    prep = ColumnTransformer([
        ("num", Pipeline([("imputar", SimpleImputer(strategy="median")), ("escalar", StandardScaler())]), num),
        ("cat", Pipeline([("imputar", SimpleImputer(strategy="most_frequent")), ("codificar", OneHotEncoder(handle_unknown="ignore"))]), cat)])
    return Pipeline([("preprocesamiento", prep), ("modelo", modelo)])

regresion_logistica = armar_pipeline(LogisticRegression(max_iter=1000)).fit(X_train, y_train)
p_rl = regresion_logistica.predict_proba(X_test)[:, 1]          # probabilidad de abandono de cada cliente de prueba
validacion = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# --- supuestos del caso y herramientas de la clase ---
COSTO_CONTACTAR, COSTO_PERDER, CAPACIDAD = 5, 60, 300

def metricas(y_real, p, umbral, c_fp=COSTO_CONTACTAR, c_fn=COSTO_PERDER):
    pred = p >= umbral
    vp = int((pred & (y_real == 1)).sum()); fp = int((pred & (y_real == 0)).sum())
    fn = int((~pred & (y_real == 1)).sum()); vn = int((~pred & (y_real == 0)).sum())
    precision = vp / (vp + fp) if vp + fp else 0.0
    recall = vp / (vp + fn) if vp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"umbral": umbral, "contactos": vp + fp, "VP": vp, "FP": fp, "FN": fn, "VN": vn,
            "precisión": precision, "recall": recall, "F1": f1, "costo": c_fp * fp + c_fn * fn}

UMBRALES = np.round(np.arange(0.02, 0.91, 0.01), 2)
def costos_por_umbral(y_real, p, c_fp=COSTO_CONTACTAR, c_fn=COSTO_PERDER):
    """Costo total para cada umbral de UMBRALES, de una vez."""
    pred = p[:, None] >= UMBRALES[None, :]
    return c_fp * (pred & (y_real[:, None] == 0)).sum(0) + c_fn * (~pred & (y_real[:, None] == 1)).sum(0)

print(f"Clientes: {len(df):,} · Entrenamiento: {len(X_train):,} · Prueba: {len(X_test):,} · AUC de la regresión logística en prueba: {roc_auc_score(y_test, p_rl):.3f}")

## Parte 2 · Tu caso

Cambiá `MI_CASO` por el número de tu equipo (1 a 7) y corré la celda. Te muestra la situación, la complicación, el pedido y lo que tienen que entregar.

In [ ]:
DESAFIOS = {
    1: dict(titulo='Capacidad limitada', situacion='El equipo comercial puede llamar a 300 clientes por cada 2.000 puntuados.', complicacion='El umbral que minimiza el costo marca más clientes de los que el equipo puede atender.', pedido='Elegí el umbral que respeta la capacidad y mostrá cuánto se pierde frente al umbral de menor costo sin límite.', entregar=['El umbral que respeta la capacidad de 300', 'Precisión y recall de esos 300 clientes', 'Su costo total frente al del umbral de menor costo, y cuántos contactos pide ese umbral'], cuidado='Elegir el umbral por costo y pasarse de la capacidad.'),
    2: dict(titulo='Contactar se vuelve caro', situacion='Una campaña de contacto pasa a costar 20 por cliente, en lugar de 5. Perder a un cliente que se iba sigue costando 60.', complicacion='Cambia el costo del error y, con él, el umbral que conviene.', pedido='Recalculá el umbral de menor costo con el costo nuevo y compará con el anterior.', entregar=['La curva de costo con el costo de 20 y con el de 5', 'El umbral recomendado y cuántos clientes se contactan', 'Cuánto costaría seguir usando el umbral anterior'], cuidado='Cambiar el costo sin recalcular el umbral.'),
    3: dict(titulo='No todos los clientes valen lo mismo', situacion='Perder un cliente cuesta 30 si es Esencial, 60 si es Plus y 120 si es Premium. Contactar sigue costando 5.', complicacion='Un único umbral trata igual a segmentos que no valen lo mismo.', pedido='Compará el mejor umbral único con un umbral para cada segmento.', entregar=['El costo total con el mejor umbral único', 'El costo total con un umbral por segmento', 'Una recomendación: ¿vale la pena la complejidad?'], cuidado='Usar el mismo costo de pérdida para todos los segmentos.'),
    4: dict(titulo='El modelo no rinde igual en todos los canales', situacion='Web, app y tienda tienen perfiles distintos, y la web concentra más clientes del segmento Esencial.', complicacion='Una métrica global puede esconder que un canal queda desatendido.', pedido='Medí precisión y recall del modelo en cada canal con el umbral 0,25 y decidí si algún canal queda desatendido.', entregar=['Una tabla con precisión, recall y porcentaje de clientes contactados por canal', 'El canal con peor desempeño y una hipótesis de por qué', 'Una decisión: qué haría el equipo si los canales difieren'], cuidado='Reportar solo la métrica global.'),
    5: dict(titulo='El árbol que memoriza', situacion='El equipo prefiere un árbol de decisión porque se explica fácil.', complicacion='A mayor profundidad, el árbol mejora en entrenamiento y empeora con datos nuevos.', pedido='Mostrá la brecha entre entrenamiento y validación según la profundidad y recomendá una.', entregar=['El AUC de entrenamiento y de validación cruzada para al menos cinco profundidades', 'La profundidad recomendada, elegida sin mirar la prueba', 'El AUC en prueba, mirado una sola vez'], cuidado='Elegir la profundidad con el conjunto de prueba.'),
    6: dict(titulo='¿Vale la pena el ensamble?', situacion='Se propone reemplazar la regresión logística por un Random Forest.', complicacion='La ganancia es chica, y el costo y la opacidad suben.', pedido='Compará los dos modelos con el mismo split y recomendá cuál usar.', entregar=['El AUC en validación cruzada y en prueba de cada modelo', 'Las bajas encontradas entre los 300 de mayor probabilidad y el tiempo de entrenamiento', 'Una recomendación que incluya la explicabilidad'], cuidado='Comparar con particiones distintas.'),
    7: dict(titulo='¿Se puede confiar en esta probabilidad?', situacion='El equipo quiere usar la probabilidad del modelo para estimar costos, y alguien propone sumar la variable motivo_baja.', complicacion='Una variable posterior al abandono infla la métrica, y una probabilidad mal calibrada distorsiona los costos.', pedido='Verificá si la probabilidad es creíble y si sumar motivo_baja es una buena idea.', entregar=['Una tabla de calibración por deciles: lo que dijo el modelo contra lo que pasó', 'El AUC con y sin motivo_baja', 'Una recomendación: ¿se puede usar la probabilidad? ¿se suma la variable?'], cuidado='Celebrar un AUC casi perfecto.'),
}

MI_CASO = 1          # <- el número de tu equipo, del 1 al 7

d = DESAFIOS[MI_CASO]
print(f"DESAFÍO {MI_CASO} · {d['titulo']}\n")
print("Situación:    ", d["situacion"])
print("Complicación: ", d["complicacion"])
print("Pedido:       ", d["pedido"])
print("\nQué entregar:")
for e in d["entregar"]: print("  •", e)
print("\nCuidado con:  ", d["cuidado"])
print("\nEn todos los casos: tabla de métricas + validación de fuga + recomendación defendible.")

## Parte 3 · Ejemplo resuelto (sobre otra pregunta)

Para que vean el recorrido completo, acá hay un ejemplo resuelto sobre **otra pregunta**: ¿cómo rinde el modelo en cada **región**, con el umbral de 0,25? No es el desafío de ningún equipo. Fíjense en tres cosas: se mide **por grupo**, se muestra el **tamaño del grupo** (`n`) y se compara con la métrica global.

In [ ]:
def metricas_por_grupo(y_real, p, grupo, umbral):
    filas = []
    for valor in sorted(pd.Series(grupo).dropna().unique()):
        m = np.asarray(grupo == valor)
        r = metricas(y_real[m], p[m], umbral)
        filas.append({"grupo": valor, "n": int(m.sum()), "tasa real de abandono": y_real[m].mean(),
                      "contactados": r["contactos"] / m.sum(), "precisión": r["precisión"], "recall": r["recall"]})
    g = metricas(y_real, p, umbral)
    filas.append({"grupo": "TOTAL", "n": len(y_real), "tasa real de abandono": y_real.mean(),
                  "contactados": g["contactos"] / len(y_real), "precisión": g["precisión"], "recall": g["recall"]})
    return pd.DataFrame(filas)

ejemplo = metricas_por_grupo(y_prueba, p_rl, X_test["region"].to_numpy(), 0.25)
ejemplo.style.format({"tasa real de abandono": "{:.1%}", "contactados": "{:.1%}", "precisión": "{:.1%}", "recall": "{:.1%}"}).hide(axis="index")

**Qué se ve en el ejemplo:** el recall cambia de una región a otra, y las regiones más chicas tienen un `n` bajo, así que sus números son menos estables. Una conclusión sobre un grupo sin su tamaño no se sostiene.

## Parte 4 · Tu turno

Resuelvan su desafío en las celdas que siguen. Las herramientas de la clase en vivo ya están cargadas: `metricas()`, `costos_por_umbral()`, `armar_pipeline()`, `p_rl` (probabilidades de la regresión logística en prueba) y `validacion`.

### Paso A · La tabla de métricas

Armen la tabla que pide su caso. Una buena tabla tiene los **números y su unidad**, y deja claro qué se está comparando.

In [ ]:
# 👉 TU TURNO · Paso A: la tabla de métricas de tu caso
# Tip: pd.DataFrame([metricas(y_prueba, p_rl, 0.25), ...]) arma una tabla con varias filas.

### Paso B · El gráfico

Un gráfico tiene que mostrar **la decisión**, no decorar. Elijan uno que se entienda sin explicación: una curva de costo, barras por grupo, dos curvas de entrenamiento y prueba, una curva de calibración.

In [ ]:
# 👉 TU TURNO · Paso B: el gráfico de tu caso

### Paso C · Validación de fuga

Todos los equipos validan que su modelo no use información posterior al abandono. La función `auc_aislado` mide cuánto predice **cada variable por sí sola**: una variable que sola separa casi perfecto (AUC cerca de 1) es sospechosa. Corran la validación sobre las variables de su modelo y sobre las que sumen.

In [ ]:
def auc_aislado(tabla, columna, objetivo="abandono_90d"):
    """AUC de una variable sola. Cerca de 0,5: no predice. Cerca de 1: sospecha de fuga."""
    valores = tabla[columna]
    if pd.api.types.is_numeric_dtype(valores):
        puntaje = valores.fillna(valores.median())
    else:                                                   # categórica: se usa la tasa de abandono de cada categoría
        valores = valores.astype(object).where(valores.notna(), "ninguno")
        puntaje = valores.map(tabla.groupby(valores)[objetivo].mean())
    auc = roc_auc_score(tabla[objetivo], puntaje)
    return max(auc, 1 - auc)

candidatas = NUM + CAT + ["motivo_baja"]
fuga = pd.Series({c: auc_aislado(df, c) for c in candidatas}).sort_values(ascending=False)
fuga.to_frame("AUC de la variable sola").style.format("{:.3f}")

**Cómo leer la tabla:** las variables que entran al modelo separan a los clientes de forma moderada (AUC entre 0,5 y 0,75). `motivo_baja` separa **perfecto**: solo existe para quien ya abandonó. Si su caso agrega alguna variable nueva, agréguenla a la lista `candidatas` y corran la celda de nuevo.

## Parte 5 · Hallazgo y ficha de evidencia

**Nuestro hallazgo** *(editen esta celda y escriban una frase: qué encontraron, con qué número y qué decisión habilita)*:

---

**Ficha de evidencia · Equipo: ________ · Caso: ________**

- **Decisión que habilita:**
- **Población y período:** clientes de prueba de NovaRetail (2.000), con corte al 31/12/2025.
- **Procedimiento:** *(qué hicieron, en orden)*
- **Resultado:** *(los números que sostienen la recomendación)*
- **Validación:** *(qué controlaron: fuga, partición, calibración, tamaño de grupo)*
- **Límite:** *(qué no pueden afirmar)*

### Si se traban

- Miren de nuevo el **ejemplo resuelto** de la Parte 3: está pensado como modelo.
- Trabajen siempre con `y_prueba` y `p_rl` para evaluar, y con `X_train` e `y_train` para elegir y entrenar.
- El umbral no lo define el modelo: lo definen el **costo** de cada error y la **capacidad**.